---



In [1]:
# Setup
%pip install python-dotenv --upgrade --quiet langchain langchain-huggingface sentence-transformers langchain-community

from dotenv import load_dotenv
load_dotenv()

import os
from langchain_huggingface import HuggingFaceEmbeddings

# Using a FREE, open-source model from Hugging Face
# 'all-MiniLM-L6-v2' is small, fast, and very good for English.
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

You should consider upgrading via the 'C:\Users\megha\AppData\Local\Programs\Python\Python39\python.exe -m pip install --upgrade pip' command.


Note: you may need to restart the kernel to use updated packages.


C:\Users\megha\AppData\Local\Programs\Python\Python39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
vector = embeddings.embed_query("Apple")

print(f"Dimensionality: {len(vector)}")
print(f"First 8 numbers: {vector[:8]}")

Dimensionality: 384
First 8 numbers: [-0.006138438358902931, 0.0310117956250906, 0.06479357928037643, 0.010941504500806332, 0.0052671851590275764, -0.04747648164629936, 0.08120306581258774, 0.028980975970625877]


In [3]:
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

vec_human = embeddings.embed_query("human")
vec_monkey = embeddings.embed_query("monkey")
vec_vanya = embeddings.embed_query("vanya")
vec_vaish = embeddings.embed_query("vaish")
vec_hiren = embeddings.embed_query("hiren")
vec_karthik = embeddings.embed_query("karthik")

print(f"human vs monkey: {cosine_similarity(vec_human, vec_monkey):.4f}")
print(f"vanya vs monkey: {cosine_similarity(vec_vanya, vec_monkey):.4f}")
print(f"monkey vs vaishnavi: {cosine_similarity(vec_monkey, vec_vaish):.4f}")
print(f"monkey vs hiren: {cosine_similarity(vec_monkey, vec_hiren):.4f}")
print(f"hiren vs vaishnavi: {cosine_similarity(vec_hiren, vec_vaish):.4f}")
print(f"vanya vs vaishnavi: {cosine_similarity(vec_vanya, vec_vaish):.4f}")
print(f"karthik vs vaishnavi: {cosine_similarity(vec_karthik, vec_vaish):.4f}")
print(f"karthik vs hiren: {cosine_similarity(vec_karthik, vec_hiren):.4f}")
print(f"monkey vs karthik: {cosine_similarity(vec_monkey, vec_karthik):.4f}")

human vs monkey: 0.4969
vanya vs monkey: 0.3131
monkey vs vaishnavi: 0.2311
monkey vs hiren: 0.2602
hiren vs vaishnavi: 0.3075
vanya vs vaishnavi: 0.3656
karthik vs vaishnavi: 0.2572
karthik vs hiren: 0.3320
monkey vs karthik: 0.1426


In [13]:
# Experiment: Let's compare "Cat", "Dog", and "Car".
vec1 = embeddings.embed_query("cat")
vec2 = embeddings.embed_query("dog")
vec3 = embeddings.embed_query("car")
print(f"cat vs dog: {cosine_similarity(vec1, vec2):.4f}")
print(f"dog vs car: {cosine_similarity(vec2, vec3):.4f}")
print(f"cat vs car: {cosine_similarity(vec1, vec3):.4f}")

cat vs dog: 0.6606
dog vs car: 0.4756
cat vs car: 0.4633


---



In [4]:
# Setup
%pip install python-dotenv --upgrade --quiet faiss-cpu langchain-huggingface sentence-transformers langchain-community langchain_google_genai
from dotenv import load_dotenv
load_dotenv()

import getpass
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key:  ")

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# Using the same free model as Part 4a
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

You should consider upgrading via the 'C:\Users\megha\AppData\Local\Programs\Python\Python39\python.exe -m pip install --upgrade pip' command.
C:\Users\megha\AppData\Local\Programs\Python\Python39\lib\site-packages\google\api_core\_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.9.13). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)


Note: you may need to restart the kernel to use updated packages.


C:\Users\megha\AppData\Local\Programs\Python\Python39\lib\site-packages\google\auth\__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
C:\Users\megha\AppData\Local\Programs\Python\Python39\lib\site-packages\google\oauth2\__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
C:\Users\megha\AppData\Local\Programs\Python\Python39\lib\site-packages\google\api_core\_python_version_support.py:246: FutureWarning: You are using a non-supported P

Enter your Google API Key:   ········


In [5]:
from langchain_core.documents import Document

docs = [
    Document(page_content="Vanya loves to eat lotte choco pie"),
    Document(page_content="The secret password to my pesu academy is 'Vaishnu@100."),
    Document(page_content="LLM halucinates more,so we use navie RAG model"),
]

In [6]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever()

In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

template = """
Answer based ONLY on the context below:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

result = chain.invoke("What is my secret password?")
print(result)

The secret password is 'Vaishnu@100.


---



In [8]:
import faiss
import numpy as np

# Mock Data: 10,000 vectors of size 128
d = 384
nb = 20000
xb = np.random.random((nb, d)).astype('float32')

In [9]:
index = faiss.IndexFlatL2(d)
index.add(xb)
print(f"Flat Index contains {index.ntotal} vectors")

Flat Index contains 20000 vectors


In [10]:
nlist = 100 # How many 'zip codes' (clusters) we want
quantizer = faiss.IndexFlatL2(d) # The calculator for distance
index_ivf = faiss.IndexIVFFlat(quantizer, d, nlist)

# We MUST train it first so it learns where the clusters are
index_ivf.train(xb)
index_ivf.add(xb)

In [11]:
M = 16 # Number of connections per node (The 'Hub' factor)
index_hnsw = faiss.IndexHNSWFlat(d, M)
index_hnsw.add(xb)

In [12]:
m = 8 # Split vector into 8 sub-vectors
index_pq = faiss.IndexPQ(d, m, 8)
index_pq.train(xb)
index_pq.add(xb)
print("PQ Compression complete. RAM usage minimized.")

PQ Compression complete. RAM usage minimized.
